# Multi-Agent Codebase Intelligence System
### A Complete Technical Walkthrough

---

This notebook is a **reading guide**. Every section explains a part of the system in plain language — what it is, why it was built that way, what it receives, what it produces, and how it connects to everything else.

By the end you will understand the full system: its architecture, its data flow, every agent, every memory store, every design decision, and every trade-off.

---

### What the system is

A production-grade multi-agent coding assistant. You upload a ZIP of any codebase, ask a question in natural language, and receive generated, fixed, or explained code that is grounded in your actual project files. The system improves with every query — it learns from reviewer feedback, stores successful patterns, and accumulates domain knowledge.

It is modeled after Cursor (project-aware generation), Devin (multi-step planning with tool use), GitHub Copilot Workspace (RAG from actual code), and LangGraph Cloud (multi-agent orchestration).

---

### Table of Contents

1. Project Structure & File Map
2. The Big Picture — System Architecture
3. The Hybrid Architecture Explained
4. Data Structures — AgentState & AgentMessage
5. Ingestion Pipeline — How Code Enters the System
6. The RAG Engine — Hybrid Retrieval
7. The Query Rewriter — 4-Layer Expansion
8. The Knowledge Base — Domain Docs as Second Brain
9. The LangGraph Pipeline — Graph Structure & Routing
10. Agent Deep Dives — Every Agent Explained
11. The ReAct Loop — Coder's Tool-Calling Mechanism
12. Self-RAG & Corrective RAG
13. The Memory System — How the System Learns
14. Agent-to-Agent Communication — The Message Bus
15. The Verification Layer
16. LangSmith Tracing
17. Complete Data Flow — One Query, Every Step
18. Pitfalls & Design Decisions
19. Switching LLMs


---
## 1. Project Structure & File Map

Every file in the system, and exactly what it is responsible for.

```
multi_agent_coder/
│
├── app/
│   ├── main.py                    FastAPI app + startup event (knowledge base ingestion)
│   │
│   ├── agents/                    All agent logic lives here
│   │   ├── state.py               AgentState TypedDict — the shared memory of the entire pipeline
│   │   ├── intent_classifier.py   Classifies query intent: write / fix / explain / general
│   │   ├── supervisor.py          Centralized router + retry decision maker
│   │   ├── planner.py             Breaks the task into ordered execution steps
│   │   ├── retriever.py           Gathers all context: RAG + tools + memory + knowledge base
│   │   ├── rag_grader.py          Self-RAG scorer + Corrective RAG reformulator
│   │   ├── coder.py               Code generator with ReAct tool loop (max 3 iterations)
│   │   ├── verifier.py            AST syntax checker — no LLM, pure Python ast module
│   │   ├── reviewer.py            Multi-dimensional code reviewer (4 criteria per intent)
│   │   └── graph.py               LangGraph StateGraph: nodes, edges, and all routing logic
│   │
│   ├── services/                  All infrastructure and data services
│   │   ├── llm.py                 LLM abstraction — Groq or Ollama, switchable in one file
│   │   ├── embeddings.py          Text → vector via nomic-embed-text through Ollama
│   │   ├── rag.py                 Hybrid BM25 + Chroma retrieval with cosine reranking
│   │   ├── bm25.py                BM25 keyword index, per-project, persisted to .pkl files
│   │   ├── chunker.py             AST-aware chunker — functions and classes as atomic units
│   │   ├── indexer.py             File walker, filters to code-only extensions
│   │   ├── ingestion.py           ZIP → extract → chunk → embed → store pipeline
│   │   ├── query_rewriter.py      4-layer query expansion
│   │   ├── knowledge.py           Domain knowledge base from ./knowledge/*.md files
│   │   ├── memory.py              Persistent experience memory in Chroma 'experience' collection
│   │   ├── session_memory.py      Short-term session memory stored as a JSON file
│   │   ├── learning.py            Extracts and stores lessons from reviewer feedback
│   │   └── tools.py               File system tools: read, search, grep, write — all sandboxed
│   │
│   └── routes/
│       └── upload.py              FastAPI endpoints + response cleaning helpers
│
├── knowledge/                     Domain knowledge docs, ingested once at startup
│   ├── numpy_guide.md
│   ├── pandas_guide.md
│   ├── sklearn_guide.md
│   └── python_patterns.md
│
├── workspace/                     Uploaded projects live here, one UUID directory per project
├── chroma_db/                     Persistent ChromaDB — 5 collections
├── bm25_indices/                  Per-project BM25 index files (.pkl)
└── session_memory.json            Short-term session notes, cleared on new project upload
```

### The 5 Chroma Collections

ChromaDB is the central persistent store. The system uses five separate collections, each serving a different purpose and written/read by different parts of the system.

| Collection | What it stores | Written by | Read by |
|---|---|---|---|
| `codebase` | Project code chunks as vector embeddings | `rag.store_chunks()` at upload | `rag.query_codebase()` on every query |
| `experience` | Mistake→fix patterns and approved code patterns | `memory.store_memory()` via learning | `memory.retrieve_memory()` via retriever |
| `knowledge_base` | Domain docs (numpy, pandas, sklearn guides) | `knowledge.ingest_knowledge_base()` at startup | `knowledge.retrieve_knowledge()` via retriever |
| `project_expansions` | Per-project token co-occurrence maps | `query_rewriter.index_chunks_for_expansion()` at upload | `query_rewriter.rewrite_query()` on every query |
| `global_patterns` | Cross-project token co-occurrence | Same as above | Same as above |


---
## 2. The Big Picture — System Architecture

The system receives a query and a project ID from the user, runs it through a 12-node LangGraph pipeline, and returns a structured response. Here is the full flow from entry to exit.

```
User
 │  query + project_id
 ▼
FastAPI /query endpoint
 │  initializes AgentState, calls graph.invoke()
 ▼
┌─────────────────────────────────────────────────────────────────────┐
│                       LangGraph StateGraph                          │
│                                                                     │
│  ┌──────────┐   ┌────────────┐   ┌──────────────────────────────┐  │
│  │  INTENT  │──▶│ SUPERVISOR │──▶│          PLANNER             │  │
│  │ classify │   │  (router)  │   │  3 ordered execution steps   │  │
│  └──────────┘   └────────────┘   └──────────────┬───────────────┘  │
│                                                  │                  │
│                                                  ▼                  │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │                        RETRIEVER                             │  │
│  │   hybrid RAG + tool search + experience memory + knowledge   │  │
│  └───────────────────────────────┬──────────────────────────────┘  │
│                                  │                                  │
│                                  ▼                                  │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │                     RAG GRADER  (Self-RAG)                   │  │
│  │         cosine similarity score — is context relevant?       │  │
│  └──────────┬─────────────────────────────────┬─────────────────┘  │
│             │ score ≥ 0.22 (ok)               │ score < 0.22 (low) │
│             │                                 ▼                    │
│             │                   ┌─────────────────────────────┐   │
│             │                   │       CORRECTIVE RAG         │   │
│             │                   │  reformulate + retry query   │   │
│             │                   └──────────────┬──────────────┘   │
│             └──────────────────────────────────┘                  │
│                                  │                                  │
│                                  ▼                                  │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │                    CODER  (ReAct loop)                        │  │
│  │   generate → tool request? → fetch → generate → final output │  │
│  └───────────────────────────────┬──────────────────────────────┘  │
│                                  │                                  │
│                                  ▼                                  │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │                  VERIFIER  (ast.parse, no LLM)               │  │
│  └──────────┬─────────────────────────────────┬─────────────────┘  │
│             │ syntax ok                       │ syntax fail         │
│             │                    increment_syntax_retry             │
│             │                         └──────────► CODER (retry)   │
│             ▼                                                       │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │                    REVIEWER  (4-dimension eval)               │  │
│  └──────────┬─────────────────────────────────┬─────────────────┘  │
│             │ APPROVED                        │ REJECTED            │
│             │                   ┌─────────────────────────────┐   │
│             │                   │      SUPERVISOR RETRY        │   │
│             │                   │  reads feedback, guides coder│   │
│             │                   └──────────────┬──────────────┘   │
│             │                                  └──► CODER (retry)  │
│             ▼                                                       │
│  ┌──────────────────────────────────────────────────────────────┐  │
│  │              LEARN  (store lessons and patterns)              │  │
│  └───────────────────────────────┬──────────────────────────────┘  │
│                                  ▼  END                             │
└─────────────────────────────────────────────────────────────────────┘
 │
 ▼
FastAPI returns: answer, plan, review, tool_calls, agent_log, confidence
```


---
## 3. The Hybrid Architecture Explained

The system uses a **hybrid centralized + decentralized architecture**. This is a standard pattern in production multi-agent systems at companies like Anthropic and DeepMind. Understanding it is key to understanding why the system is designed the way it is.

### The Centralized Part — Supervisor

The `supervisor` node is the single routing authority. Nothing proceeds without the supervisor's decision. It decides the path (proceed / skip retrieval / needs more context) based on intent and query. After a rejection, `supervisor_retry` reads the reviewer's specific feedback and generates targeted guidance for the coder's retry. This centralization ensures every query follows a coherent strategy and retries have direction.

### The Decentralized Part — Agents on the Message Bus

Once routed, agents operate independently. They do not call each other. They do not import each other. Each agent reads the **shared message bus** — `state["messages"]` — to understand what previous agents decided and found, then appends its own structured message. The coder never calls the retriever. It reads what the retriever put on the bus.

This is the blackboard pattern in AI systems: a shared workspace where agents read and write without coupling to each other. Adding a new agent means adding a new node to the graph that reads the bus and appends to it. Nothing else changes.

### Why not purely centralized?

A fully centralized system would have the supervisor call every agent directly and pass results between them. This creates tight coupling, makes the supervisor a massive god-function, and makes the whole thing fragile — one agent change breaks everything.

### Why not purely decentralized?

A fully decentralized system with no supervisor would have agents deciding their own routing. This makes the pipeline hard to reason about, hard to debug, and makes retry logic complex. Who decides when to retry? Which agent has enough information to make that call?

The hybrid gives you the best of both: clear authority (supervisor routes), loose coupling (message bus communication), independent agent logic, and full traceability.


---
## 4. Data Structures — AgentState & AgentMessage

### AgentState

`AgentState` is a `TypedDict` defined in `app/agents/state.py`. It is the single shared object that LangGraph passes through every node. Every agent reads from it and writes back to it. Think of it as the pipeline's memory — the complete picture of everything that has happened and everything that is needed.

Here is every field, what it holds, and who owns it:

**Core input** — set once at the API layer, never changed:
- `project_id` — UUID of the uploaded project. Used by every retrieval and tool call to stay within project boundaries.
- `query` — the user's raw question or task.

**Agent communication bus** — the most important field:
- `messages` — a growing list of `AgentMessage` objects. Every agent appends to this. Downstream agents read it to understand the pipeline context. This is how agents communicate.

**Routing** — set by intent and supervisor, read by everything downstream:
- `intent` — one of `write`, `fix`, `explain`, `general`. Controls which system prompts are used, which retrieval strategy runs, which review criteria are applied.
- `route` — one of `proceed`, `skip_retrieval`, `needs_more_context`. Set by the supervisor, read by the retriever and planner.

**Planning** — set by planner, read by coder:
- `plan` — raw LLM plan text. Cleaned and displayed in the UI plan tab.
- `plan_steps` — parsed list of step strings. The coder uses this to display "EXECUTE NOW: Step 1" in its prompt.
- `current_step` — which step the coder is currently executing. Increments when the coder makes a tool request.
- `file_structure` — compact project overview used by the planner to know what files exist.

**Retrieval** — set by retriever and rag_grader, read by coder and reviewer:
- `context` — a dict with two keys: `code` (up to 7 context chunks, including knowledge base content) and `memory` (formatted past mistakes from the experience collection).
- `retrieval_confidence` — cosine similarity mean score between query and top chunks. Shown as the RAG % badge in the UI.
- `retrieval_ok` — True if confidence >= 0.22. Controls whether corrective RAG fires and injects a low-confidence note into the coder prompt.
- `corrective_attempted` — prevents the corrective RAG loop from running twice.

**Tool tracking** — written by retriever and coder, shown in tools tab:
- `tool_calls` — plain-language log of every tool invoked, e.g. `"grep_search('pipeline') → 21 hits"`.
- `tool_results` — raw result dicts keyed by tool name, available for debugging.

**ReAct loop** — the coder's mid-task tool-requesting state:
- `coder_tool_requests` — list of tool requests the coder made mid-generation.
- `coder_iterations` — how many LLM calls the coder has made. Capped at 3.
- `additional_context` — context fetched by the coder's tool calls, injected into subsequent iterations.

**Generation** — the outputs:
- `answer` — the coder's raw output. Gets overwritten if a retry happens.
- `final_answer` — the polished answer sent to the user. Set by the reviewer on approval or by supervisor_retry on acceptance.

**Syntax verification:**
- `syntax_ok` — True if `ast.parse()` passed on all extracted code blocks.
- `syntax_error` — the exact error string from `ast.parse()`, e.g. `"SyntaxError at line 18: expected an indented block after function definition on line 1"`. Injected directly into the coder's retry prompt.

**Review and retry:**
- `review` — full reviewer LLM output. Cleaned at the API boundary before sending to UI.
- `needs_retry` — True if the reviewer said REJECTED.
- `retry_count` — how many review retries have happened. When this reaches 1, the reviewer auto-accepts.

**Supervisor intelligence:**
- `supervisor_notes` — on first pass, stores the raw routing decision. On retry, stores the specific coder guidance that `supervisor_retry` generated from the reviewer's feedback. The coder reads this as `retry_note` in its prompt.

---

### AgentMessage

Every agent appends one `AgentMessage` to the bus when it finishes. The structure is simple: `agent` (who sent it), `type` (what kind of message), `content` (human-readable summary, truncated to 300 chars), and `metadata` (structured data like scores and counts).

Here is what the bus looks like after a complete successful run:

```
[intent_classifier / decision]
  content:  "Query intent classified as: write"
  metadata: {intent: "write"}

[supervisor / decision]
  content:  "Routing decision: proceed"
  metadata: {route: "proceed", raw: "PROCEED"}

[planner / plan]
  content:  "1. Load csv file using pandas  2. Define preprocess function  3. Return DataFrame"
  metadata: {intent: "write", step_count: 3, steps: [...]}

[retriever / context]
  content:  "Retrieved 6 context pieces. Tools: query_codebase, grep_search, read_full_file"
  metadata: {chunk_count: 6, tool_count: 3, memory_count: 0}

[rag_grader / decision]
  content:  "Retrieval confidence: 0.477 (OK)"
  metadata: {score: 0.477, ok: True}

[coder / code]
  content:  "import pandas as pd

def load_and_preprocess(filepath: str)..."
  metadata: {intent: "write", iterations: 1, tool_requests: 0, retry_count: 0}

[verifier / decision]
  content:  "Syntax verified OK — 2 Python block(s) passed"
  metadata: {block_count: 2}

[reviewer / review]
  content:  "APPROVED: ..."
  metadata: {needs_retry: False, intent: "write", syntax_ok: True}
```

Content is truncated to 300 characters in the bus. The full coder output is stored separately in `state["answer"]`. The bus carries summaries, not full outputs — this prevents the bus from consuming thousands of characters of the coder's context window on downstream prompts.


---
## 5. Ingestion Pipeline — How Code Enters the System

When a user uploads a ZIP file, a complete ingestion pipeline runs before any query can be answered. This pipeline is what makes the system project-aware.

```
POST /upload-zip
 │
 ├── extract_zip(zip_path, workspace/{project_id}/)
 │     unzips the uploaded archive into a UUID-named directory
 │
 ├── session_memory.clear()
 │     new project means fresh session — previous lessons don't leak across projects
 │
 ├── process_project(project_path)                          [ingestion.py]
 │    │
 │    ├── indexer.get_code_files()                          [indexer.py]
 │    │     walks all subdirectories, filters by extension
 │    │     allowed: .py .js .ts .tsx .jsx .cpp .c .h .java
 │    │
 │    └── for each file:
 │           chunk_code(content, file_path)                 [chunker.py]
 │
 │                IF .py file:
 │                  ast.parse(content) → walk all nodes
 │                  for each FunctionDef / AsyncFunctionDef / ClassDef:
 │                    extract line_start to end_lineno
 │                    if block > 80 lines: split into sub-chunks
 │                    → {content, file, type="function"/"class", name=function_name}
 │
 │                ELSE (any other language — fallback):
 │                  split every 80 lines
 │                  → {content, file, type="generic", name="chunk_N"}
 │
 ├── store_chunks(chunks, project_id)                       [rag.py]
 │    │
 │    ├── for each chunk:
 │    │     get_embedding(text)     nomic-embed-text via Ollama → float vector
 │    │     chroma.add(
 │    │         id = "{project_id}_{i}",
 │    │         document = chunk text,
 │    │         embedding = vector,
 │    │         metadata = {file, project, chunk_id}
 │    │     )
 │    │
 │    ├── BM25Index.build(chunks)                           [bm25.py]
 │    │     builds IDF-weighted keyword index over all chunks
 │    │     persists to bm25_indices/{project_id}.pkl
 │    │
 │    └── index_chunks_for_expansion(chunks, project_id)    [query_rewriter.py]
 │          sliding window (±5 tokens) over every chunk
 │          counts how often each token co-occurs with neighbors
 │          upserts to Chroma 'project_expansions' (per-project)
 │          merges into Chroma 'global_patterns' (cross-project)
 │
 └── response: {project_id, chunks_count, status, files}
```

### Why AST-based chunking?

A regular text splitter cuts code every N characters or lines. This means a 200-line function gets split at line 80, giving you an incomplete function that starts at a random `if` statement. That chunk is useless for retrieval — there's no context for what it belongs to.

AST-based chunking treats each **function** and each **class** as an atomic unit. When the retriever finds that `one_iteration()` is relevant to a query, it returns the entire function — all the logic, all the variable names, the full picture. This is what makes the coder's context genuinely useful.

### Why BM25 and Chroma together?

**BM25** is a keyword search algorithm. It finds chunks that contain the exact words in the query. It excels at specific identifiers — if you ask about `family_type`, BM25 finds every chunk that mentions `family_type`. It struggles with synonyms and semantic similarity.

**Chroma** does semantic vector search. It finds chunks that are conceptually similar to the query even if they use completely different words. "impute missing values" might retrieve chunks containing `fillna()` and `dropna()` even though none of those words appeared in the query.

Combined, deduplicated, and reranked by cosine similarity: you get better recall than either approach alone. The BM25 results come first in the merge (preserving keyword precision at the top), and then everything is reranked semantically.


---
## 6. The RAG Engine — Hybrid Retrieval

`app/services/rag.py` — called on every query. The central retrieval function.

```
query_codebase(query, project_id, n_results=5)
 │
 ├── Step 1: Query Rewriting
 │     rewrite_query(query, project_id)
 │     returns expanded query with related technical terms
 │     example: "how to load csv" → "how to load csv pandas read_csv csv reader file"
 │
 ├── Step 2: BM25 Keyword Search
 │     load BM25 index from .pkl (cached in memory after first load)
 │     search rewritten query, top 8 results
 │     good for: exact function names, library identifiers, variable names
 │
 ├── Step 3: Chroma Semantic Search
 │     embed the rewritten query → float vector
 │     collection.query(
 │         query_embeddings = [vector],
 │         n_results = 8,
 │         where = {"project": project_id},    ← project isolation
 │         include = ["documents", "embeddings"]
 │     )
 │     good for: conceptually related code, synonyms, patterns
 │
 ├── Step 4: Deduplication
 │     merge BM25 results + Chroma results
 │     remove exact duplicates, BM25 results come first
 │     typically 4–8 unique candidates
 │
 └── Step 5: Cosine Reranking
       get query embedding (already computed in step 3)
       for each candidate: use cached Chroma embedding if available, else embed
       sort all candidates by cosine(query_embedding, chunk_embedding) descending
       return top n_results
```

### The BM25 Scoring Formula

BM25 stands for Best Match 25. The score for a document `d` given query `q` is:

```
Score(d, q) = Σ  IDF(t) × [ tf(t,d) × (k1 + 1) ]  /  [ tf(t,d) + k1 × (1 − b + b × |d| / avgdl) ]

where:
  t       = each term in the query
  IDF(t)  = log( (N − df(t) + 0.5) / (df(t) + 0.5) + 1 )
              N    = total documents in index
              df(t) = number of documents containing term t
              → rare terms (high IDF) score higher than common ones
  tf(t,d) = frequency of term t in document d
  k1 = 1.5   saturation: 10 occurrences is not 10× better than 1
  b  = 0.75  length normalization: short docs not unfairly penalized
  avgdl    = average document length across the index
```

In plain terms: a chunk that specifically mentions the exact identifier you searched for scores highly, especially if that identifier is rare across the codebase. Common words like `import` or `return` are heavily penalized by IDF.

### Project Isolation

Every Chroma query includes `where={"project": project_id}`. All projects share a single `codebase` collection. Without this filter, uploading a Flask project and then uploading a NumPy project would have them contaminating each other's retrieval. The project UUID is the partition key.


---
## 7. The Query Rewriter — 4-Layer Expansion

`app/services/query_rewriter.py`

Short natural language queries are poor retrieval queries. "how to load csv" does not contain `read_csv`, `DataFrame`, or `pd` — the identifiers that actually appear in code. The query rewriter expands the query with related technical terms before it reaches BM25 and Chroma.

```
rewrite_query("how to load csv file in pandas", project_id)
 │
 ├── Layer 1: Static Vocabulary  (instant, zero cost)
 │     17 keyword → term-expansion mappings
 │     "pandas" → "pd DataFrame read_csv groupby merge apply"
 │     "numpy"  → "np array ndarray reshape zeros ones dtype"
 │     "auth"   → "login authenticate token jwt password user"
 │     If any keyword found in query → append its expansion
 │     Result so far: "how to load csv file in pandas pd DataFrame read_csv..."
 │
 ├── Layer 2: Global Co-occurrence  (instant, from Chroma 'global_patterns')
 │     For the first 2 tokens of the query: look up most frequent neighbors
 │     These are learned from every project ever uploaded to this system
 │     Gets richer with every new project — the system improves over time
 │     Appends top 4 neighbors
 │
 ├── Layer 3: Project Co-occurrence  (instant, from Chroma 'project_expansions')
 │     For tokens in the query: look up neighbors in THIS project's index
 │     Built from the actual codebase at ingestion time
 │     If your project uses "df" near "col" and "loc" → those get appended
 │     This is why "dot product" queries expanded to include "col df_new loc"
 │     Those variable names came from dataprepro.py's actual code
 │
 └── Layer 4: LLM Semantic Expansion  (1 LLM call, validated)
       Prompt: "Python programming terms related to: {query}. List 4, one per line."
       Output validated:
         → reject if > 8 words (model rambling)
         → reject if contains prompt-leak phrases (model echoed its own prompt)
         → reject if identical to original query
       Appends if valid, silently skips on failure or timeout
       
Final: join all parts, split into tokens, take first 20 words
```

### How the Co-occurrence Index is Built

At ingestion time, for every chunk in the uploaded project, the rewriter applies a sliding window:

```
tokens = tokenize(chunk.content)
for each token at position i:
    window = tokens[i−5 : i+6]        ← 5 before and 5 after
    for each neighbor in window:
        project_learned[token][neighbor] += 1
```

The top 10 most frequent neighbors per token are stored in `project_expansions` for that project. They are also merged into `global_patterns`, which accumulates co-occurrence knowledge across every project ever uploaded. Upload more projects → global patterns get richer → query expansion gets smarter across the whole system.

This is why querying about "pipeline" on a project that uses scikit-learn will automatically expand with pipeline-related terms that actually appear in that codebase.


---
## 8. The Knowledge Base — Domain Docs as Second Brain

`app/services/knowledge.py` + `./knowledge/*.md`

The system doesn't only know what's in the uploaded project. It also has domain knowledge about the libraries those projects commonly use. This lives in the `./knowledge/` directory as markdown files.

### What's in the knowledge base

- **numpy_guide.md** — array creation, common operations (dot, sum, mean), indexing, broadcasting, and the most frequent numpy mistakes
- **pandas_guide.md** — loading data, selecting columns, groupby, merge, fillna, and common mistakes like chained indexing
- **sklearn_guide.md** — Pipeline construction, train_test_split, evaluation metrics, and mistakes like fitting the scaler on full data before splitting
- **python_patterns.md** — function templates with type hints and docstrings, error handling patterns, common Python mistakes

### Ingestion flow

At server startup, `ingest_knowledge_base()` runs once:

```
Walk ./knowledge/ for .md, .txt, .rst files
For each file not already in Chroma:
    chunk by paragraph boundaries (~600 chars, 100-char overlap)
    for each chunk:
        embed with nomic-embed-text
        add to knowledge_collection with metadata={source: filename}

Dedup: checks existing IDs by prefix — won't re-ingest on restart
```

### How it reaches the coder

In `retriever_node`, step 5 calls `retrieve_knowledge(query, k=2)`. This does a cosine similarity search against the `knowledge_base` Chroma collection and returns the 2 most relevant chunks. They are labeled `"# From knowledge base:
{content}"` and merged into the context alongside project code.

### Why this matters

Without the knowledge base, asking "write a numpy dot product function" gives the coder only the uploaded project's code as context — which may not contain any numpy examples at all. With the knowledge base, the retriever also finds the numpy guide's section on `np.dot()`, and the coder has real documentation to draw from when generating code.


---
## 9. The LangGraph Pipeline — Graph Structure & Routing

`app/agents/graph.py`

LangGraph works by defining nodes (functions that transform state) and edges (connections between nodes, some conditional). The graph is compiled once at startup and reused for every query.

### The 12 Nodes

| Node | Function | File |
|---|---|---|
| `intent` | `intent_node()` | intent_classifier.py |
| `supervisor` | `supervisor_node()` | supervisor.py |
| `planner` | `planner_node()` | planner.py |
| `retriever` | `retriever_node()` | retriever.py |
| `rag_grader` | `rag_grader_node()` | rag_grader.py |
| `corrective_rag` | `corrective_rag_node()` | rag_grader.py |
| `coder` | `coder_node()` | coder.py |
| `verifier` | `verifier_node()` | verifier.py |
| `increment_syntax_retry` | pure state mutation | graph.py |
| `reviewer` | `reviewer_node()` | reviewer.py |
| `supervisor_retry` | `supervisor_retry_node()` | supervisor.py |
| `learn` | `learning_node()` | graph.py + learning.py |

### Fixed Edges (always run in order)

```
START → intent → supervisor → planner → retriever → rag_grader
corrective_rag → coder → verifier
increment_syntax_retry → coder
learn → END
```

### Conditional Edges (routing logic)

**After supervisor:** Both paths go to planner, but the `route` flag in state differs. If route is `skip_retrieval`, the retriever skips the actual retrieval and the planner notes this. The conditional is effectively communicating a flag, not choosing a different destination.

**After rag_grader:** If confidence is below 0.22 and corrective RAG hasn't run yet → `corrective_rag`. Otherwise → `coder`.

**After verifier:** If syntax failed and fewer than 1 syntax retry has occurred → `increment_syntax_retry` → `coder`. If syntax failed but max retries hit → `reviewer` anyway (give up, let reviewer handle it). If syntax passed → `reviewer`.

**After reviewer:** If `needs_retry` is True → `supervisor_retry`. Otherwise → `learn`.

**After supervisor_retry:** If `needs_retry` is still True and `retry_count < 1` → `coder`. Otherwise → `learn`.

### Possible Execution Paths

| Path | Nodes visited | Approximate LLM calls |
|---|---|---|
| Happy path | 10 | 5–6 |
| + syntax retry | 12 | 6–7 |
| + review rejected | 13 | 7–8 |
| + corrective RAG | 11 | 5–6 |
| Worst case (all retries) | ~17 | 9–10 |

### Recursion Limit

`graph.invoke(state, {"recursion_limit": 40})` — the graph has cycles (coder → verifier → coder, reviewer → supervisor_retry → coder). Without a recursion limit, a malfunctioning LLM could loop until memory exhaustion. 40 comfortably covers the worst-case path with buffer.


---
## 10. Agent Deep Dives — Every Agent Explained

### Intent Classifier

**Receives:** `state["query"]`

**Does:** First tries regex pattern matching against lists of write/fix/explain patterns. Confidence = number of pattern matches. If confidence is zero or the query is three words or fewer (too short to classify by pattern), it falls back to a lightweight LLM call asking for a single-word classification. The LLM output is validated — only the exact words `write`, `fix`, `explain`, or `general` are accepted.

**Why regex first:** An LLM call for intent classification is wasteful when the query contains "write" or "fix" — the answer is obvious from the words. The LLM fallback handles ambiguous short queries like "pandas csv" or "broken auth" where a human would also need to think about intent.

**Writes:** `state["intent"]`

---

### Supervisor

**Receives:** `state["intent"]`, `state["query"]`, the last 6 messages from the bus

**Does:** Builds a compact string from recent bus messages (what has the pipeline done so far?) and uses an LLM call to decide the routing: PROCEED, NEEDS_MORE_CONTEXT, or SKIP_RETRIEVAL. Normalizes the output — "SKIP" anywhere in the response triggers skip_retrieval.

**When SKIP_RETRIEVAL triggers:** General coding questions with no project-specific keywords (no "in this project", "this file", "our codebase", etc.). The retriever still runs but marks the context as skipped.

**Writes:** `state["route"]`, `state["supervisor_notes"]`

---

### Supervisor Retry

**Receives:** `state["answer"]`, `state["review"]`, `state["query"]`, `state["retry_count"]`

**Does:** This is where intelligent retry guidance happens. It reads the reviewer's specific rejection reasons and generates 2–3 concrete instructions for the coder: "Add import pandas as pd at top", "Add docstring explaining parameters and return type". This guidance is stored in `supervisor_notes` and injected into the coder's retry prompt. The coder's retry is not blind — it knows exactly what was wrong and what to fix.

**Writes:** `state["needs_retry"]`, `state["retry_count"]`, `state["supervisor_notes"]`

---

### Planner

**Receives:** `state["intent"]`, `state["route"]`, `state["project_id"]`

**Does:** Calls `get_project_summary()` which reads the workspace directory, lists files with line counts, and extracts function/class names from Python files using AST. This summary is injected into the system prompt so the planner knows what actually exists in the project. Then an LLM call produces 3 numbered steps. `_parse_steps()` quality-filters the output: skips code blocks, skips prose lines that start with "I " or "The ", skips lines over 150 characters, enforces contiguous step numbering, caps at 6 steps.

**Writes:** `state["plan"]`, `state["plan_steps"]`, `state["current_step"] = 0`, `state["file_structure"]`

---

### Retriever

**Receives:** `state["query"]`, `state["project_id"]`, `state["intent"]`, `state["route"]`, `state["messages"]`

**Does:** This is the most tool-intensive agent in the pipeline. It runs 5 sub-steps:

1. Reads the planner's message from the bus to understand what files and identifiers to search for
2. Runs `query_codebase()` for semantic + keyword retrieval (unless route is skip_retrieval)
3. For `fix` and `explain` intents: extracts code identifiers from the query and plan, runs `search_code()` (AST-level), reads the defining file with `read_full_file()`
4. For `write` intent: extracts what the user wants to write, runs `grep_search()` to find related patterns, reads the file with the most hits
5. Retrieves past mistakes from `retrieve_memory()` and domain knowledge from `retrieve_knowledge()`

All context is merged, deduplicated, and capped at 7 chunks. No LLM calls — this is pure tool use.

**Writes:** `state["context"]`, `state["tool_calls"]`, `state["tool_results"]`

---

### RAG Grader

**Receives:** `state["context"]["code"]`, `state["query"]`

**Does:** Embeds the query. Embeds the top 3 context chunks. Computes cosine similarity between the query vector and each chunk vector. Takes the mean as `retrieval_confidence`. Marks `retrieval_ok = True` if confidence >= 0.22. If route is skip_retrieval, sets confidence to 1.0 immediately and returns — no point scoring context that wasn't retrieved.

**Writes:** `state["retrieval_confidence"]`, `state["retrieval_ok"]`

---

### Corrective RAG

**Receives:** `state["query"]`, `state["intent"]`, `state["project_id"]`

**Does:** Builds an intent-specific prefix (for `write`: "function class definition example similar", for `fix`: "bug error exception handler fix", etc.) and prepends it to the original query. Re-runs `query_codebase()` with this reformulated query. If new chunks are found, replaces the context. Sets `corrective_attempted = True` to prevent re-running.

**Why this helps:** If retrieval confidence is low, it usually means the query is too abstract. Prefixing it with intent-specific code vocabulary steers the semantic search toward more relevant code patterns.

**Writes:** `state["context"]`, `state["retrieval_confidence"]`, `state["corrective_attempted"] = True`

---

### Coder

**Receives:** Essentially everything in state — context, plan, memory, session notes, retry feedback

**Does:** The ReAct loop. Explained in full in Section 11.

**Writes:** `state["answer"]`, `state["coder_iterations"]`, `state["tool_calls"]`, `state["additional_context"]`

---

### Verifier

**Receives:** `state["answer"]`, `state["intent"]`

**Does:** Explained in full in Section 15.

**Writes:** `state["syntax_ok"]`, `state["syntax_error"]`

---

### Reviewer

**Receives:** `state["answer"]`, `state["query"]`, `state["intent"]`, context chunks, recent bus messages

**Does:** Guards: if answer is empty, skip. If retry_count >= 1, auto-accept (prevents infinite loop). Reads recent bus messages from planner, retriever, rag_grader, and verifier to understand the pipeline context. Applies 4-dimension criteria per intent type:

- **write:** correctness, completeness (imports, error handling), style (follows codebase patterns), quality (docstring, readability)
- **fix:** addresses the described bug, shows corrected code, doesn't introduce regressions, accurate explanation
- **explain:** grounded in actual code (not generic), references real function/class names, no hallucinated behavior, clear structure
- **general:** answers the actual question, grounded in codebase, no hallucinations, specific and actionable

Produces APPROVED or REJECTED with detailed reasoning. On APPROVED, uses the coder's answer directly — reviewer rewrites of approved answers tend to be worse than the original with smaller models. On REJECTED, extracts the reviewer's corrected version using `_clean_reviewer_body()`.

**Writes:** `state["review"]`, `state["needs_retry"]`, `state["final_answer"]`

---

### Learning Node

**Receives:** `state["review"]`, `state["answer"]`, `state["intent"]`, `state["query"]`

**Does:** Explained in full in Section 13.

**Writes:** Chroma `experience` collection + `session_memory.json`


---
## 11. The ReAct Loop — Coder's Tool-Calling Mechanism

**ReAct = Reason → Act → Observe → repeat**

The coder is not limited to one-shot generation. It can realize mid-task that it needs more information, request it, receive the result, and continue generating. This is how Cursor and Devin work at their core.

### The Loop

```
for i in range(MAX_CODER_ITERATIONS = 3):
    iterations += 1

    BUILD PROMPT:
      if syntax_error exists (syntax retry path):
          MINIMAL PROMPT — just the error, the broken code, and the task
          no context, no plan, no session notes
          this prevents weak models from echoing the full context back
          forces focus on the specific syntax error
      else (normal path):
          FULL PROMPT — system persona + pipeline status + execution plan
          + codebase context (up to 2000 chars) + past mistakes
          + session history (last 5 notes) + retry guidance if applicable

    output = call_llm(prompt)
    output = _clean_coder_output(output)    extract from ```python fences if present

    tool_request = _parse_tool_request(output)
        looks for:  NEED_FILE: auth.py
                    NEED_SEARCH: authenticate_user

    _is_placeholder(value):
        rejects: NEED_FILE: <relative_path>      template value, model didn't fill it in
        rejects: NEED_FILE: filename.py           generic placeholder
        rejects: any value without "." or "/"    not a real file path
        accepts: NEED_FILE: dataprepro.py
        accepts: NEED_SEARCH: one_iteration

    if valid tool_request AND iterations < 3:
        result = _execute_tool_request(project_id, request)
            NEED_FILE  → read_full_file() → returns full file content
            NEED_SEARCH → search_code() (AST-level definitions first) + grep fallback
        additional_context.append(result)
        tool_calls.append("coder→read_full_file('auth.py')")
        state["current_step"] += 1
        continue    ← loop again with enriched context
    else:
        final_answer = output
        break
```

### Why tool validation matters

Without placeholder validation, a weak model outputs `NEED_FILE: <relative_path>` verbatim, which gets passed to `read_full_file()` as a literal path, which fails with an error. The placeholder detection prevents this: if the value looks like an unfilled template, the tool request is silently discarded and the current output is treated as the final answer.

### What the execution plan does in the coder prompt

The plan steps from the planner are formatted as:

```
Execution plan:
→ EXECUTE NOW: Step 1: Load the csv file using pandas
  Pending: Step 2: Define preprocessing function
  Pending: Step 3: Return cleaned DataFrame
```

When the coder makes a tool request and additional context is fetched, `current_step` increments. On the next iteration the plan shows Step 2 as "EXECUTE NOW". This gives the coder a sense of progress through the task.

### Example execution

**Query:** "fix the bug in the authenticate function"  
**Iteration 1:** LLM outputs `NEED_FILE: auth.py` — it can't fix what it can't see.  
**Tool execution:** `read_full_file("auth.py")` → 200 lines of actual authentication code  
**Iteration 2:** LLM now has the full function. Outputs corrected code grounded in the real implementation.  
**Result:** 2 iterations, 1 tool request, coder saw the actual bug before writing the fix.


---
## 12. Self-RAG & Corrective RAG

Two techniques from 2023/2024 AI research, implemented in `rag_grader.py`.

### Self-RAG — Scoring Your Own Retrieval

**The problem:** Just because the retriever returned 6 chunks doesn't mean they're relevant. If someone asks "how do I take the dot product" and the project has no numpy code, the retrieved chunks might be about pandas DataFrames — technically similar in the embedding space but not actually useful for the question.

**Self-RAG solution:** Before passing context to the coder, compute a relevance score.

```
q_emb = get_embedding(state["query"])
for chunk in context["code"][:3]:
    c_emb = get_embedding(chunk[:400])
    score = cosine(q_emb, c_emb)
retrieval_confidence = mean(scores)
retrieval_ok = (retrieval_confidence >= 0.22)
```

The threshold of **0.22** was tuned empirically:
- Completely unrelated technical texts: ~0.10–0.18
- Related but imperfect (similar domain, different concepts): ~0.22–0.40
- Closely related (query about the exact code in context): ~0.40+

The confidence score appears as the "RAG 48%" badge in the UI. Green means ok, red means corrective RAG fired.

### Corrective RAG — Reformulating When Retrieval Fails

**The problem:** Confidence below 0.22 means the retrieved chunks are likely not useful. Passing bad context to the coder produces hallucinated answers.

**Corrective RAG solution:** Instead of giving up, reformulate the query with intent-specific vocabulary and try again.

```
prefix map:
  write   → "function class definition example similar"
  fix     → "bug error exception handler fix"
  explain → "implementation logic flow structure"
  general → "code usage pattern"

reformulated = f"{prefix} {original_query}"
new_chunks = query_codebase(reformulated, project_id)
```

The prefix steers the semantic search toward code patterns relevant to the intent type. For a `write` query, adding "function class definition" biases retrieval toward definition-heavy chunks. For a `fix` query, "bug error exception handler" biases toward error-handling code.

If the new results are better, context is updated. `corrective_attempted = True` prevents a second run and infinite loops.


---
## 13. The Memory System — How the System Learns

The system has three distinct memory stores at different time scales, plus two learned index stores in Chroma.

### Tier 1: Session Memory  (`session_memory.py` + `session_memory.json`)

**Time scale:** current session. Survives server restarts. Cleared when a new project is uploaded.

**Written by:** `learning_node` — on every query, regardless of whether it was approved or rejected.

**Read by:** `coder_node` — injected as "Session history" in the coder prompt (last 5 entries).

**Contents look like:**
```
"Wrote: def family_type(num):"
"Previously answered: from sklearn.pipeline import Pipeline"
"Answered 'how to load csv': import pandas as pd
df = pd.read_csv..."
"For 'sklearn pipeline': answer should reference actual Pipeline class"
```

**Why it matters:** The second query in a session knows what the first one covered. This is why — in testing — the second query about sklearn pipelines incorporated `family_type()` from the uploaded project: the session memory noted "Wrote: def family_type(num):" from the first query, and the coder saw it in the "Session history" section and reasoned about it.

---

### Tier 2: Experience Memory  (`memory.py` → Chroma `experience` collection)

**Time scale:** permanent. Survives everything. Never cleared.

**Written by:** `learning_node` — when REJECTED or APPROVED/ACCEPTED.

**Read by:** `retriever_node` → `retrieve_memory(query, k=2)` → injected into `context["memory"]` → shown to coder as "Past mistakes to avoid".

**What gets stored:**

When a query is **REJECTED**, the learning node tries these approaches in order:
1. Look for structured labels `Mistake:`, `Fix:`, `Lesson:` in the reviewer output — works with well-behaved models
2. Extract the first numbered line from the reviewer's prose feedback — works with any model that produces numbered lists
3. Store the reviewer's numbered critique items directly as patterns

When a query is **APPROVED** or **ACCEPTED** and intent is write or fix:
- Regex-finds complete function definitions in the answer
- Stores them as positive patterns: `"Pattern for write: load_and_preprocess"` + the full function body

This means the system accumulates two kinds of knowledge: what went wrong (mistake→fix pairs), and what worked (successful function patterns). Both get retrieved for similar future queries.

---

### Tier 3: Project Code  (Chroma `codebase` collection)

**Time scale:** permanent. Written at upload, read on every query.

This is the project's code as vectors. Queried by `query_codebase()` on every retrieval. Per-project isolated using the `where={"project": project_id}` filter.

---

### The Learning Loop in Detail

```
process_learning(review_text, answer, intent, query):

Determine outcome:
  is_approved = "APPROVED" in review OR "ACCEPTED" in review
  is_rejected = "REJECTED" in review

IF REJECTED:
  Try structured labels first (Mistake:/Fix:/Lesson:)
  If not found: extract first numbered line from reviewer prose
  If nothing: store reviewer's numbered items as patterns

IF APPROVED or ACCEPTED:
  If intent is write or fix:
      find complete function definitions in answer
      store as approved patterns in experience
  Add session note: "Previously answered: {first line}"

ALWAYS (regardless of outcome):
  If answer contains def/class line: "Wrote: def function_name(..."
  Else: "Answered '{query}': {first 60 chars of answer}"
  session_memory.add(note)
```

The "ALWAYS" block is why session memory accumulates even for general queries and even when the reviewer accepted on retry. A previous version only stored on explicit APPROVED, missing all retry-accepted answers. The fix was `is_approved = "APPROVED" in review OR "ACCEPTED" in review`.


---
## 14. Agent-to-Agent Communication — The Message Bus

### Why not direct function calls?

The naive implementation of a multi-agent system has agents calling each other:
```
supervisor.decide() → planner.plan(supervisor_output) → retriever.retrieve(plan)
```

This creates tight coupling. Changing the planner's output format breaks the retriever. Adding a new agent between planner and retriever requires editing both. Tracing what happened is difficult — the call stack is implicit.

### The message bus pattern

Every agent reads `state["messages"]` at the start and appends an `AgentMessage` at the end. No agent imports any other agent. The bus is append-only. The graph controls which agents run and in what order — agents themselves only know about shared state.

### Specific inter-agent communications that matter

**Planner → Retriever:**  
The retriever reads the planner's message from the bus and calls `_extract_terms()` on the plan text. If the plan says "load data from dataprepro.py", the retriever extracts "dataprepro" and "load" as search targets and runs `search_code("dataprepro")`. Without reading the planner's message, the retriever would only search based on the raw query.

**Retriever → Coder:**  
The coder reads `rag_grader` metadata for the confidence score and `retriever` metadata for the chunk count. It builds a compact "Pipeline status: 6 chunks, confidence: 0.477" header in its prompt. The coder knows the quality of its context before generating. If confidence is low, the coder also receives a `confidence_note` warning it that the context may not be directly relevant.

**Verifier → Coder (on syntax retry):**  
The verifier appends a message with type `"tool_result"` and content `"SYNTAX ERROR in generated code: SyntaxError at line 18: expected an indented block. Fix this before proceeding."`. The graph routes back to the coder, which detects `syntax_error` in state and switches to the minimal prompt path — just the exact error, the broken code, and the task. No other context. This forces the model to focus only on fixing the specific issue.

**Reviewer → Coder (via supervisor_retry):**  
The reviewer writes `REJECTED` with numbered reasons. The supervisor_retry reads those reasons, generates 2–3 concrete instructions ("Add type hints: filepath: str → pd.DataFrame", "Add docstring explaining what the function does"), and stores them in `supervisor_notes`. The coder's retry prompt includes: "Previous attempt REJECTED. Supervisor guidance: {instructions}. Reviewer said: {feedback}." The retry is not blind — the coder knows exactly what was wrong.

**All agents → Reviewer:**  
The reviewer reads the last 6 messages from planner, retriever, rag_grader, and verifier before judging. It knows: what the plan said, how many context pieces were retrieved, what the confidence score was, whether syntax was verified. This context helps the reviewer judge whether the coder had good resources to work with or was operating with limited context.


---
## 15. The Verification Layer

`app/agents/verifier.py` — sits between coder and reviewer. Zero LLM cost.

### What it does

```
verifier_node(state):

Guard: if intent is explain or general → pass through immediately
  These produce prose. Running ast.parse on an explanation is meaningless.

Guard: if answer is empty → pass through

Extract Python code blocks (in priority order):
  1. ```python ... ``` fenced blocks (most common, markdown code blocks)
  2. ``` ... ``` fenced blocks of any language
  3. If text starts with "def " / "class " / "import " / "from " / "#" → raw Python
  If nothing found → log "No code blocks found — skipping" and pass through

For each extracted block:
  ast.parse(block)
  → success: no problem
  → SyntaxError: collect "SyntaxError at line {lineno}: {msg}"

If any errors:
  state["syntax_ok"] = False
  state["syntax_error"] = "SyntaxError at line 1: unexpected indent; ..."
  append to message bus: "SYNTAX ERROR in generated code: {detail}. Fix before proceeding."
  add to session_memory: "Syntax error in generated code: {detail}"
  → graph routes: increment_syntax_retry → coder

If all blocks pass:
  state["syntax_ok"] = True
  append to message bus: "Syntax verified OK — N Python block(s) passed"
  → graph routes: reviewer
```

### Why this matters more than it looks

Without the verifier, a syntactically broken function reaches the reviewer, which makes an expensive LLM call to evaluate code that doesn't even parse. The reviewer might say "looks good to me" because it's evaluating the intent, not running the code. The user receives a syntax error they'd only discover when actually running the output.

With the verifier, broken code is caught for free (Python's built-in `ast.parse` costs nothing). More importantly, the coder receives the **exact error message** on retry — "SyntaxError at line 18: expected an indented block after function definition on line 1" is infinitely more useful than a reviewer saying "this code has a problem."

The minimal prompt on syntax retry is also important: the coder gets only the error, the broken code, and the task. No 2000-character context window. This is what prevents weak models from echoing the full context back instead of just fixing the indentation.

### increment_syntax_retry

This node runs before the coder on a syntax retry. It:
- Increments `syntax_retry_count`
- Resets `syntax_ok = None` and `syntax_error = None` so the verifier runs fresh
- Resets `coder_iterations = 0` so the coder gets a fresh ReAct loop

Maximum syntax retries is 1. If the coder fails syntax twice, the graph proceeds to the reviewer anyway. Better to let a human-readable review happen than loop forever.


---
## 16. LangSmith Tracing

Every significant operation in the system is decorated with `@traceable` from LangSmith. This gives complete observability — you can see exactly what happened inside any query.

### Coverage

**Agents (run_type=chain):**
- `intent_agent` — intent_classifier.py
- `supervisor_agent` — supervisor.py
- `supervisor_retry_agent` — supervisor.py
- `planner_agent` — planner.py
- `retriver_agent` — retriever.py
- `rag_grader_agent` — rag_grader.py
- `corrective_rag_agent` — rag_grader.py
- `coder_agent` — coder.py
- `verifier_agent` — verifier.py
- `reviewer_agent` — reviewer.py
- `learning_node` — graph.py

**LLM calls (run_type=llm):**
- `llm_call` — every `call_llm()` invocation in llm.py

**Retrieval (run_type=retriever):**
- `query_codebase` — rag.py

**Tools (run_type=tool):**
- `rewrite_query` — query_rewriter.py
- `store_memory` — memory.py
- `retrieve_memory` — memory.py

### What you see in LangSmith

For any query, you get a full trace tree showing the wall time of each node, each LLM call with exact prompt and exact response, each retrieval with the rewritten query and number of results, and each tool call with arguments and result.

This lets you answer questions like: why did corrective RAG fire for this query? What exactly did the supervisor tell the coder on retry? What context did the reviewer see when it rejected? Which queries resulted in stored experience memories?

### Required environment variables

```
LANGCHAIN_TRACING_V2=true
LANGCHAIN_API_KEY=<your LangSmith key>
LANGCHAIN_PROJECT=multi-agent-coder
```


---
## 17. Complete Data Flow — One Query, Every Step

Tracing a single query through the entire system with actual values at each step.

**Query:** "write a function to load and preprocess a CSV file"  
**Project:** dataprepro.py (contains `family_type()` and `one_iteration()`)

---

### Step 1 — Intent Classification

The query contains `write`, matching one write pattern. Confidence = 1. No LLM needed.

State after: `intent = "write"`  
Bus: `"Query intent classified as: write"`

---

### Step 2 — Supervisor

Last 6 bus messages: just the intent message. LLM sees the query and intent, outputs "PROCEED".

State after: `route = "proceed"`  
Bus: `"Routing decision: proceed"`

---

### Step 3 — Planner

`get_project_summary()` reads the workspace and finds: `"dataprepro.py (45 lines) → def family_type(), def one_iteration()"`. This goes into the system prompt so the planner knows what exists.

LLM produces:
```
1. Load csv file using pandas in dataprepro.py
2. Define preprocess function following project patterns
3. Return cleaned DataFrame from the function
```

`_parse_steps()` accepts all 3 (concrete actions, no code, ≤ 150 chars each).

State after: `plan_steps = ["Load csv...", "Define preprocess...", "Return..."]`, `current_step = 0`

---

### Step 4 — Retriever

**Sub-step 4a — Semantic RAG:**  
`rewrite_query()` expands: "csv" triggers the static vocab → appends "pandas read_csv csv reader file". Result: "write function load preprocess csv pandas read_csv csv reader file"

BM25 finds the `one_iteration()` chunk (it has preprocessing logic). Chroma finds `family_type()` and `one_iteration()` chunks. After dedup and rerank: 4 chunks from the project.

**Sub-step 4b — Write-specific tool use:**  
`_extract_write_targets()` finds ["load", "preprocess"] from the query. `grep_search("load")` gets 2 hits. `read_full_file("dataprepro.py")` returns the full 45-line file. This gets appended to extra_context.

**Sub-step 4c — Memory:**  
`retrieve_memory()` → empty (fresh system, no experience yet).

**Sub-step 4d — Knowledge base:**  
`retrieve_knowledge()` finds pandas_guide.md's "Loading data" section: `df = pd.read_csv('file.csv')` and related patterns.

**Sub-step 4e — Merge:**  
`[4 RAG chunks] + [full file content] + [pandas guide chunk]` → deduped → 6 unique context pieces.

State after: `context["code"] = [6 chunks]`, `tool_calls = ["query_codebase(...)", "grep_search('load') → 2 hits", "read_full_file('dataprepro.py')", "retrieve_knowledge(...) → 2 chunks"]`

---

### Step 5 — RAG Grader

Cosine scores for top 3 chunks: 0.43, 0.45, 0.51. Mean = 0.463. Above 0.22.

State after: `retrieval_confidence = 0.463`, `retrieval_ok = True`  
Graph: → coder (corrective RAG skipped)

---

### Step 6 — Coder (Iteration 1)

The system prompt is: "You are an expert software developer. Write clean, complete, working code."

The human message includes:
- Pipeline status: 6 chunks, confidence 0.463
- Execution plan with Step 1 as "EXECUTE NOW"
- The 6 context chunks (2000 char limit)
- Past mistakes: None
- Session history: (empty — first query)
- Task: "write a function to load and preprocess a CSV file"

LLM output:
```python
import pandas as pd

def load_and_preprocess(filepath):
    df = pd.read_csv(filepath)
    df = df.dropna()
    return df
```

No `NEED_FILE:` or `NEED_SEARCH:` in output. `final_answer = output`. Break.

State after: `answer = "import pandas as pd

def load_and_preprocess..."`, `coder_iterations = 1`

---

### Step 7 — Verifier

`_extract_python_blocks()` finds one block. `ast.parse()` succeeds.

State after: `syntax_ok = True`  
Graph: → reviewer

---

### Step 8 — Reviewer

retry_count = 0, so full review runs. Reads recent bus messages for context. Applies write criteria:

1. Correctness: loads and preprocesses? ✓
2. Completeness: has import ✓, but no type hints and no docstring ✗
3. Style: follows project patterns? Partial ✓
4. Quality: readable ✓, documented ✗

LLM output:
```
REJECTED
2. Completeness: Missing type hints and docstring
Corrected version:
import pandas as pd

def load_and_preprocess(filepath: str) -> pd.DataFrame:
    '''Load CSV and remove rows with missing values.'''
    df = pd.read_csv(filepath)
    df = df.dropna()
    return df
```

State after: `needs_retry = True`, `review = "REJECTED
2. Completeness..."`  
Graph: → supervisor_retry

---

### Step 9 — Supervisor Retry

LLM reads the reviewer's feedback and generates:
```
RETRY
Add type hints: filepath: str -> pd.DataFrame
Add a docstring explaining what the function does and what it returns
```

State after: `needs_retry = True`, `retry_count = 1`, `supervisor_notes = "Add type hints...
Add a docstring..."`  
Graph: → coder

---

### Step 10 — Coder (Iteration 2, retry)

The human message now includes:
```
Previous attempt REJECTED.
Supervisor guidance: Add type hints: filepath: str -> pd.DataFrame
                     Add a docstring explaining what the function does and returns
Reviewer said: REJECTED 2. Completeness: Missing type hints and docstring
```

LLM now produces a function with type hints and a full docstring. No tool request.

State after: `answer = improved version`, `coder_iterations = 2`

---

### Step 11 — Verifier (second pass)

`ast.parse()` passes. `syntax_ok = True`. → reviewer

---

### Step 12 — Reviewer (second pass)

`retry_count = 1 >= 1` → auto-accept. No LLM call.

State after: `final_answer = answer`, `review = "Accepted after retry."`  
Graph: → learn

---

### Step 13 — Learning

`review = "Accepted after retry."` → `is_approved = True` (contains "ACCEPTED")

Intent is "write", so regex finds the function definition in the answer and calls:
```
store_memory(
    mistake = "Pattern for write: load_and_preprocess",
    fix = "def load_and_preprocess(filepath: str) -> pd.DataFrame:
    ..."
)
```

Session memory gets:
```
"Previously answered: import pandas as pd"
"Wrote: def load_and_preprocess(filepath: str) -> pd.DataFrame:"
```

---

### Final Response to User

```
answer:               complete function with type hints and docstring
plan:                 "1. Load csv...
2. Define...
3. Return..."
review:               "Accepted after retry."
intent:               "write"
retrieval_confidence: 0.463
tool_calls:           ["query_codebase(...)", "grep_search(...)", "read_full_file(...)", "retrieve_knowledge(...)"]
coder_iterations:     2
syntax_ok:            True
```


---
## 18. Pitfalls & Design Decisions

The real lessons from building this system — what was tried, what went wrong, what was decided and why.

---

**Sequential vs parallel agents**

Decision: sequential. Coding has a forced dependency chain. You cannot retrieve before knowing intent. You cannot code before retrieving. You cannot review before coding. Parallel agents would simply block on each other. This is the correct architecture for coding tasks — unlike research tasks where multiple subtopics can be investigated simultaneously. The temptation to add parallel agents is real but wrong.

---

**LLM call minimization**

Decision: use non-LLM paths wherever reasoning isn't genuinely needed. The verifier uses `ast.parse` (free). The retriever uses tool calls (free). Corrective RAG reformulates without an LLM call. Only agents where reasoning actually matters — planner, coder, reviewer, supervisor retry — use the main model. This matters especially on Groq's free tier where 6–8 LLM calls per query triggers 429 rate limiting.

---

**Model-agnostic architecture with isolated patches**

Decision: design for production, patch only where necessary for weak models. The tinyllama patches (minimal syntax retry prompt, prompt-leak detection in the query rewriter, supervisor rule-based fallback) are isolated behind specific conditions. They don't change the architecture — they only activate when the specific problem they solve is detected. Swapping to a better model means removing those conditions, not rewriting the system.

---

**Output validation at every boundary**

Decision: always validate and clean LLM output before using it. Weak models echo their input prompts back as output. Without output validation, the "answer" shown to the user contains the entire prompt with instructions visible. The query rewriter has `_PROMPT_LEAK_PHRASES` detection. The reviewer has `_NOISE_MARKERS` to stop extraction at echo lines. The coder has `_clean_coder_output()` to strip prose wrapping around code. The API layer has `_extract_plan_steps()` and `_trim_review()` as a final cleaning pass before the UI sees anything.

---

**Retrieval confidence threshold at 0.22**

Decision: 0.22 cosine similarity as the floor for "retrieval is useful." Below this, corrective RAG fires. The value was tuned empirically: unrelated technical texts score ~0.10–0.18 against code queries. Related but imperfect matches: ~0.22–0.40. Closely related: ~0.40+. The dataprepro.py test project consistently scored 0.43–0.51 for relevant queries, confirming 0.22 is the right floor for this embedding model (nomic-embed-text).

---

**Project isolation in Chroma**

Decision: `where={"project": project_id}` on every single Chroma query. All projects share one `codebase` collection. Without the filter, uploading a Flask API project and then a NumPy data pipeline would have their chunks mixed in every retrieval. The UUID partition key is the only thing preventing cross-project contamination. This filter must be present on every new query path — forgetting it is one of the easiest bugs to introduce when extending the system.

---

**Session memory cleared on new project upload**

Decision: `session_memory.clear()` is called in `upload_zip()`. A lesson learned while working on a pandas preprocessing project ("always include type hints") would otherwise appear as "Session history" when working on a React frontend project. The session is scoped to one project at a time.

---

**Hard limits on all retry loops**

Decision: `MAX_CODER_ITERATIONS = 3`, `MAX_SYNTAX_RETRIES = 1`, reviewer auto-accepts at `retry_count >= 1`, `recursion_limit = 40`. Without these, a malfunctioning LLM can produce infinite loops. The graph has true cycles — coder → verifier → coder, reviewer → supervisor_retry → coder. Without the recursion limit in `graph.invoke()`, these cycles run until the process runs out of memory.

---

**Message bus content truncation**

Decision: agent message content is truncated to 300 characters in the bus. The full coder output (1000+ characters of code) doesn't need to live in every downstream agent's context. The bus carries summaries. Without truncation, after several agents have appended their outputs the bus grows to 10,000+ characters — consuming most of the coder's context window with agent chatter instead of actual code context.

---

**AST chunking with line-based fallback**

Decision: AST-first for Python, line-based for everything else. AST parsing fails on files with syntax errors and on all non-Python files. The fallback handles both cases silently. The chunker never raises an exception — it always returns something, even if it's just 80-line generic chunks.

---

**Learning from retry-accepted answers**

This was a bug that persisted until fixed: the original `process_learning()` only stored lessons on `"APPROVED" in review_text`. But when the reviewer auto-accepts on retry, it writes `"Accepted after retry."` — which contains neither APPROVED nor REJECTED. So the learning loop never fired for retry-accepted answers, which are the majority of answers once the system has been used for a few queries. The fix: `is_approved = "APPROVED" in review OR "ACCEPTED" in review`.


---
## 19. Switching LLMs — What Changes, What Doesn't

The system was deliberately designed so that swapping the LLM requires changing exactly one file: `app/services/llm.py`.

### How to switch from Groq to local Ollama

Comment out the Groq block and uncomment the Ollama block. Change `MODEL = "tinyllama"`. Change `call_llm_raw` to use direct HTTP calls to the Ollama API instead of LangChain. Nothing else in the system changes — no agent files, no graph, no retrieval, no memory.

### What improves automatically with a better model

When you run `llama-3.1-8b-instant` (Groq) instead of tinyllama:
- The planner outputs real numbered steps instead of full tutorial code
- The supervisor LLM call produces meaningful PROCEED/SKIP routing decisions
- The query rewriter LLM layer contributes actual semantic terms
- The reviewer catches real bugs and missing features with specific feedback
- The coder follows style patterns from the context it receives
- The learning loop can extract structured `Mistake:` / `Fix:` labels from reviewer output
- The supervisor retry generates targeted actionable guidance
- The ReAct `NEED_FILE:` requests use real file names instead of placeholder templates

### What stays identical regardless of model

Everything infrastructure-related is model-independent:
- AST-based chunking
- BM25 keyword search
- Chroma semantic search and cosine reranking
- Query rewriter layers 1, 2, and 3 (static vocab, global co-occurrence, project co-occurrence)
- Self-RAG cosine confidence scoring
- Syntax verification via `ast.parse()`
- Knowledge base ingestion and retrieval
- Session memory read and write
- Experience memory store and retrieve
- Project isolation via the `where` filter
- All retry caps and recursion limits
- LangSmith tracing
- All UI output cleaning functions

### The tinyllama patches and when to remove them

Three patches exist specifically for tinyllama and can be relaxed or removed when upgrading:

1. **`coder.py` minimal syntax retry prompt** — the full context causes tinyllama to echo the prompt. A better model handles full context on syntax retry, so the minimal path can be removed or the condition loosened.

2. **`query_rewriter.py` prompt-leak detection** — the `_PROMPT_LEAK_PHRASES` check exists because tinyllama echoes its own prompt into the expansion output. A better model outputs only the expansion terms.

3. **`reviewer.py` noise marker filtering** — `_NOISE_MARKERS` stops extraction when the reviewer echoes prompt content. Better models don't echo their prompts, so this filtering becomes unnecessary.

### Upgrade path

| Level | Model | Notes |
|---|---|---|
| 1 | tinyllama (local Ollama) | Free, slow, poor output quality |
| 2 | llama-3.1-8b-instant (Groq) | Free tier, fast, good quality — what this system was tested with |
| 3 | llama3.2:3b or qwen2.5-coder:3b (local Ollama) | Better local option, requires ~4GB RAM |
| 4 | qwen2.5-coder:7b (local Ollama) | Excellent for code, requires ~8GB+ VRAM |
| 5 | GPT-4o-mini or Claude Haiku (paid API) | Best quality, paid per token |

---

## Summary

```
Architecture:     Hybrid Centralized + Decentralized (LangGraph StateGraph)
Nodes:            12 (intent → supervisor → planner → retriever → rag_grader
                      → corrective → coder → verifier → reviewer
                      → supervisor_retry → learn → END)

Retrieval:        BM25 keyword + Chroma semantic + cosine reranking
                  Self-RAG scoring + Corrective RAG reformulation
                  4-layer query expansion (static + global + project + LLM)
                  Domain knowledge base from ./knowledge/*.md

Generation:       ReAct loop (max 3 iterations, NEED_FILE / NEED_SEARCH)
                  Intent-aware system prompts
                  Supervisor retry with targeted coder guidance from reviewer feedback

Verification:     ast.parse() syntax check, zero LLM cost
                  Exact error injected into minimal retry prompt

Review:           4-dimension criteria per intent type
                  Auto-accept at retry_count >= 1

Memory:           Session (JSON, last 5 notes, cleared per project upload)
                  Experience (Chroma, permanent, mistake→fix + approved patterns)
                  Project code (Chroma, per-project isolated)
                  Query expansion (Chroma, learned co-occurrence, improves over time)
                  Domain knowledge base (Chroma, library guides)

Communication:    Typed AgentState TypedDict as shared blackboard
                  AgentMessage bus (append-only, every agent reads)
                  supervisor_notes carries targeted retry guidance to coder

Tracing:          LangSmith @traceable on all 11 agents + 4 service functions

LLM:              Switchable in one file (llm.py)
                  Architecture fully model-agnostic
                  Weak-model patches isolated and removable
```
